In [ ]:
import pandas as pd
from jax import numpy as jnp
from jax.random import PRNGKey

pd.options.plotting.backend = "plotly"

from summer3.graph import CompartmentValues, defer, Parameter
from summer3.epi import TransitionFlow, Stratification, CompartmentMap, CompartmentalEpiModel, CategoryData, CompartmentalModelODE, build_istate, dti_to_epoch

In [ ]:
start_time = 0
end_time = 70
model_comps = [
    "vaccinated", 
    "susceptible", 
    "infectious_asympt", 
    "infectious_sympt", 
    "recovered",
]
infect_comps = [
    "infectious_asympt", 
    "infectious_sympt",
]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)

In [ ]:
def frequency_dependent_transmission(compartment_values, infectious_comps, contact_rate):
    infectious_pop = compartment_values.query(infectious_comps).sum()
    total_pop = compartment_values.sum()
    return contact_rate * infectious_pop / total_pop

progression = TransitionFlow(
    "progression",
    disease_state["infectious_asympt"],
    disease_state["infectious_sympt"],
    1.0 / Parameter("asympt_time", 0.0),
)
recovery = TransitionFlow(
    "recovery",
    disease_state["infectious_asympt"],
    disease_state["recovered"],
    1.0 / Parameter("recovery_time", 0.0),
)

times = range(start_time, end_time)
sir_model = CompartmentalEpiModel(humans, times)
init_pops = CategoryData(disease_state.categories(), jnp.array(([0.0, 0.99, 0.01, 0.0, 0.0])))
sir_model.set_initial_population(init_pops)

sir_model.add_flow(progression)
sir_model.add_flow(recovery)

print(
    f"The simulation runs from time {round(start_time)} to {round(end_time)}. "
    f"The model compartments are called: {', '.join(model_comps)}. "
    f"Of these compartments, {', '.join(infect_comps)} are considered equally infectious."
)

In [ ]:
data = pd.Series(
    {
        20.0: 187.0,
        22.0: 302.0,
        24.0: 982.0,
        26.0: 1383.0,
        28.0: 2203.0,
        30.0: 4304.0,
        32.0: 5479.0,
        34.0: 12945.0,
        36.0: 11310.0,
        38.0: 14238.0,
        40.0: 9262.0,
        42.0: 3677.0,
        44.0: 2899.0,
        46.0: 2149.0,
        48.0: 1443.0,
        50.0: 1049.0,
    },
)

In [ ]:
mask_cov = Parameter("face_mask_coverage", 0.0)
mask_effic = Parameter("face_mask_efficacy", 0.0)
inf_rate = Parameter("contact_rate", 0.0) * (1.0 - mask_cov * mask_effic)
src = "susceptible"
dest = "infectious_asympt"

force_of_infection = defer(frequency_dependent_transmission)(
    CompartmentValues, 
    disease_state[infect_comps], 
    inf_rate,
)

infection = TransitionFlow(
    "infection", 
    disease_state["susceptible"], 
    disease_state["infectious_asympt"], 
    force_of_infection,
)
sir_model.add_flow(infection)

print(
    f"The process of infection transitions people from the {src} compartment to the {dest} compartment. "
    f"The rate of infection is determined by the coverage '{mask_cov.key}' and efficacy '{mask_effic.key}' of face masks. "
    "The product of these two parameters determine how much face masks reduce transmission by."
)

In [ ]:
mask_effic.key

In [ ]:
params = {"contact_rate": 0.5, "asympt_time": 4.0, "recovery_time": 4.0, "face_mask_efficacy": 0.0, "face_mask_coverage": 0.0}
results = sir_model.run(params)

In [ ]:
results["compartments"].to_pandas_df().plot()